# Mastermind Solver 🎯
Guess a secret code of 4 pegs in 6 colours (1,296 possibilities); after each guess you learn how many pegs are the right colour in the right place (black) and the right colour in the wrong place (white). In 1977 Donald Knuth showed that the **minimax** strategy (choose the guess whose *worst-case* answer leaves the fewest candidates) always wins in ≤ 5 guesses. Rebuild it.

Background: [[Information Theory]] (compare with [[Project - Wordle Solver]]) · overview: [[Fun Projects]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import collections, itertools, numpy as np
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
COLORS, PEGS = 6, 4
ALL = list(itertools.product(range(COLORS), repeat=PEGS))
print(len(ALL), "possible codes")

## 1. Feedback
Black = positions where guess and secret agree. White = (number of colours they have in common, counted with multiplicity) − black. Return `(black, white)`.

In [ ]:
def feedback(guess, secret):
    # TODO 1: black and white pegs
    raise NotImplementedError  # TODO 1

## 2. Knuth's minimax guess
Codes are now numbered 0…1295 and `FB[g, s]` holds the feedback of guess `g` for secret `s`, encoded as `5·black + white` (precomputed with your `feedback`). For every possible guess `g` (not only the remaining candidates!), count how the remaining `candidates` (an index array) split by feedback (`np.bincount`); the *worst case* is the largest group. Return the `g` with the smallest worst case, preferring one in `cand_set` on ties (else the lowest index).

In [ ]:
def knuth_guess(candidates, cand_set):
    # TODO 2: minimise the size of the largest feedback group
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_mm():
    return feedback((0, 0, 1, 1), (0, 1, 2, 1)) == (2, 1) and feedback((0, 0, 0, 0), (1, 1, 1, 1)) == (0, 0) and feedback((1, 2, 3, 4), (4, 3, 2, 1)) == (0, 4) and \
        max(collections.Counter(feedback((0, 0, 1, 1), s) for s in ALL).values()) == 256
check("mastermind feedback", _t_mm, "(0,0,1,1) vs (0,1,2,1): 2 black, 1 white; Knuth's opener 0011 leaves at most 256 codes.")
if ready("mastermind feedback"):
    FB = np.array([[5 * b_ + w for b_, w in (feedback(g, s) for s in ALL)] for g in ALL], dtype=np.int64)    # 1296 × 1296 table, ~3 s
def solve(secret, verbose=False):
    cands = np.arange(len(ALL)); guess, n, target = ALL.index((0, 0, 1, 1)), 1, ALL.index(secret)            # Knuth's opening guess 0011
    while True:
        fb = FB[guess, target]
        if verbose: print(f"  guess {n}: {''.join('RGBYOP'[c] for c in ALL[guess])} → {fb // 5} black, {fb % 5} white ({len(cands)} candidates before)")
        if fb == 5 * PEGS: return n
        cands = cands[FB[guess, cands] == fb]
        guess = int(cands[0]) if len(cands) == 1 else knuth_guess(cands, set(cands.tolist())); n += 1
if ready("mastermind feedback"):
    try:
        print("secret RGBY:"); solve((0, 1, 2, 3), verbose=True)
        rng = np.random.default_rng(0); sample = [ALL[i] for i in rng.choice(len(ALL), 150, replace=False)]
        turns = [solve(s) for s in sample]; print(f"\n150 random secrets: average {np.mean(turns):.2f} guesses, worst {max(turns)}, distribution {dict(sorted(collections.Counter(turns).items()))}")
        check("Knuth's bound", lambda: max(turns) <= 5, "Knuth's minimax strategy must solve every code in at most 5 guesses.")
    except NotImplementedError:
        print("❌ Knuth's bound: not implemented yet – fill in the TODO above.")

<details><summary>▶ Solution</summary>

```python
def feedback(guess, secret):
    black = sum(g == s for g, s in zip(guess, secret))
    common = sum(min(guess.count(c), secret.count(c)) for c in range(COLORS))
    return black, common - black
```

```python
def knuth_guess(candidates, cand_set):
    worst = np.array([np.bincount(FB[g, candidates], minlength=25).max() for g in range(len(ALL))])
    best = np.flatnonzero(worst == worst.min())
    in_cands = [g for g in best if g in cand_set]
    return int(in_cands[0]) if in_cands else int(best[0])
```
</details>

## Stretch goals
- Swap minimax for **maximum entropy** (as in the Wordle solver). Lower average? Same worst case?
- Run all 1,296 secrets and reproduce Knuth's average of ≈ 4.48 guesses.
- 5 pegs × 8 colours: does the brute-force minimax still run in reasonable time?